# Exploración: reseñas de vinos (TP2)

Cuaderno de trabajo. El entregable es `analisis_exploratorio_vinos.py`; acá queda
el rastro del diagnóstico y de las verificaciones que llevaron a cada decisión.

| Bloque | Contenido |
|---|---|
| 0 | Diagnóstico del archivo y reconstrucción |
| 1 | Perfilado, duplicados y resumen de categóricas |
| 2 | Estadística descriptiva |
| 3 | Faltantes: mecanismo e imputación |
| 4 | Atípicos: detección y naturaleza |
| 5 | Comparativo |

## Bloque 0. Diagnóstico del archivo

La consigna pide `winemag-data-130k-v2.csv`. Lo entregado es un `.xlsx` de 21 MB.
La regla de arranque: cargarlo de la forma obvia y verificar el resultado, en vez
de darlo por bueno.

In [1]:
import io, re, csv
from collections import Counter
import pandas as pd
import numpy as np

RUTA = 'Entregable 2 - Documentación extra.xlsx'
crudo = pd.read_excel(RUTA)
crudo.head(3)

,",country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,"0,Italy,""Aromas include tropical fruit, broom,...",NaN,NaN,NaN,NaN
1,"1,Portugal,""This is ripe and fruity, a wine th...",NaN,NaN,NaN,NaN
2,"2,US,""Tart and snappy, the flavors of lime fle...",NaN,NaN,NaN,NaN


`read_excel` no se quejó, y sin embargo el resultado está mal: esperábamos 13
columnas y hay 5. Una tiene la línea CSV entera y las otras cuatro tienen nombre
genérico.

Que no haya excepción no prueba que haya cargado bien.

In [2]:
crudo.info()

<class 'pandas.DataFrame'>
RangeIndex: 129975 entries, 0 to 129974
Data columns (total 5 columns):
 #   Column                                                                                                                           Non-Null Count   Dtype
---  ------                                                                                                                           --------------   -----
 0   ,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery  129975 non-null  str  
 1   Unnamed: 1                                                                                                                       2956 non-null    str  
 2   Unnamed: 2                                                                                                                       159 non-null     str  
 3   Unnamed: 3                                                                                                     

Las cuatro columnas `Unnamed` tienen datos, y su conteo cae en cascada:
2.956, luego 159, luego 12, luego 1.

Ese decrecimiento de un orden de magnitud por columna es la firma de un texto
partido por un separador. La mayoría de las líneas no lo contiene, unas pocas lo
contienen una vez y muy pocas dos.

In [3]:
# Una fila afectada, celda por celda y sin truncar
primera = crudo.loc[crudo['Unnamed: 1'].notna()].iloc[0]
for celda in primera:
    print(celda)

48,US,"This bottling resembles the New Zealand paradigm of Sauvignon Blanc, bearing aromas of grapefruit, passion fruit and kiwi
 a sprinkling of graham cracker adds interest. The wine hits the palate like a fleshy fist, with an intense, grassy gooseberry flavor that provides plenty of punch. Pair with apricot-glazed roasted chicken.",,86,16.0,Virginia,Monticello,,,,Trump 2011 Sauvignon Blanc (Monticello),Sauvignon Blanc,Trump
nan
nan
nan


El corte cae en mitad de una oración: `...passion fruit and kiwi` y después
`a sprinkling of graham cracker...`. Falta un signo de puntuación entre las dos
partes, y ese signo es el delimitador que Excel se comió.

No es la coma: el resto de la línea conserva decenas de comas intactas. Si la
coma fuera el delimitador, esta fila estaría partida en unos 15 pedazos.

In [4]:
# El delimitador es el signo ausente: Excel se comió todas sus apariciones.
columna = crudo[crudo.columns[0]]
for ch in [',', '.', ';', ':', '|', '\t']:
    print(repr(ch), columna.str.count(re.escape(ch)).sum())

',' 2096660


'.' 490443
';' 0


':' 1626


'|' 1
'\t' 0


Dos candidatos dan cero, `;` y el tabulador, pero sólo uno de los dos es raro.

Que no haya ningún tabulador en prosa es lo normal: nadie escribe tabuladores
dentro de una nota de cata, sea o no el delimitador. Que no haya ningún punto y
coma en 130.000 reseñas escritas por críticos profesionales es imposible por
azar. Ese es el signo que alguien se llevó.

Coincide además con la evidencia gramatical del corte, porque `;` es el único
signo que une dos cláusulas independientes con minúscula a continuación. Dos
caminos distintos, misma respuesta.

In [5]:
# Reconstrucción: reponer el ';' que Excel consumió.
# El encabezado quedó como nombre de columna y hay que devolverlo al texto.
lineas = [crudo.columns[0]]
for fila in crudo.itertuples(index=False):
    lineas.append(';'.join(c for c in fila if isinstance(c, str)))

# Segundo problema: 4 reseñas con saltos de línea quedaron repartidas en
# varias filas. Toda línea sana empieza con el índice numérico y una coma.
lineas_ok = [lineas[0]]
for linea in lineas[1:]:
    if re.match(r'^\d+,', linea):
        lineas_ok.append(linea)
    else:
        lineas_ok[-1] += '\n' + linea

texto = '\n'.join(lineas_ok)
print(f'{len(lineas)} líneas -> {len(lineas_ok)} tras reunir las partidas')

129976 líneas -> 129972 tras reunir las partidas


### Criterio de verificación

Se fija antes de cargar, no después. El encabezado declara 13 nombres de columna
precedidos por un campo de índice sin nombre, así que toda línea reconstruida
tiene que dar 14 campos al parsearse como CSV.

Se usa `csv.reader` y no `.split(',')` porque las descripciones están llenas de
comas dentro de comillas.

In [6]:
print(Counter(len(f) for f in csv.reader(io.StringIO(texto))))

Counter({14: 129972})


Un solo valor, 14, para las 129.972 líneas (129.971 reseñas más el encabezado). La reconstrucción es correcta.

### Tercer problema: codificación

`Gewürztraminer` aparece como `GewÃ¼rztraminer`. Es mojibake: bytes UTF-8 leídos
como cp1252. La vuelta atrás es exacta, se recodifica a cp1252 para recuperar los
bytes y se decodifica como UTF-8.

Tres celdas pasaron dos veces por la conversión equivocada, así que el arreglo se
repite hasta que el texto deja de cambiar.

In [7]:
def reparar(texto):
    if not isinstance(texto, str):
        return texto
    for _ in range(5):
        try:
            candidato = texto.encode('cp1252').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return texto      # lo irreversible se deja intacto
        if candidato == texto:
            return texto
        texto = candidato
    return texto

roto = lambda s: isinstance(s, str) and bool(re.search(r'[ÃÂ][\x80-\xbf\xa0-\xff]', s))

df = pd.read_csv(io.StringIO(texto), index_col=0)
antes = sum(df[c].map(roto).sum() for c in df.select_dtypes(exclude='number'))
for c in df.select_dtypes(exclude='number').columns:
    df[c] = df[c].map(reparar)
despues = sum(df[c].map(roto).sum() for c in df.select_dtypes(exclude='number'))
print(f'celdas con mojibake: {antes:,} -> {despues}')
print(df.shape)

celdas con mojibake: 106,611 -> 132
(129971, 13)


Todo esto quedó consolidado en `analisis_exploratorio_vinos.py`. De acá en
adelante se importa de ahí, para no mantener dos versiones de la misma lógica.

In [8]:
from analisis_exploratorio_vinos import (
    cargar_datos, perfil_columnas, resumen_categoricas, quitar_duplicados,
    diagnostico_faltantes, validar_imputacion_precio, tratar_faltantes,
    detectar_atipicos, atipicos_contextuales, impacto_atipicos,
    extraer_anio, efecto_catador, analisis_temporal,
    agregar_longitud, analisis_longitud, mejor_relacion_calidad_precio)

df = cargar_datos()
df.shape

(129971, 13)

## Bloque 1. Perfilado, duplicados y categóricas

El enunciado dice 10 columnas y 130.000 reseñas. El archivo tiene 13 y 129.971.
Los datos son la evidencia; el enunciado es una afirmación a verificar.

In [9]:
perfil_columnas(df)

,tipo,no_nulos,nulos,nulos_%,unicos,cardinalidad_%
region_2,str,50511,79460,61.14,17,0.01
designation,str,92506,37465,28.83,37979,29.22
taster_twitter_handle,str,98758,31213,24.02,15,0.01
taster_name,str,103727,26244,20.19,19,0.01
region_1,str,108724,21247,16.35,1229,0.95
price,float64,120975,8996,6.92,390,0.30
province,str,129908,63,0.05,425,0.33
country,str,129908,63,0.05,43,0.03
description,str,129971,0,0.00,119955,92.29
points,int64,129971,0,0.00,21,0.02


`description` tiene 92,3% de valores únicos y `title` 91,4%. Que no lleguen al
100% delata filas repetidas.

In [10]:
print('duplicados exactos (13 columnas):', df.duplicated().sum())
print('por description + title      :', df.duplicated(subset=['description','title']).sum())

duplicados exactos (13 columnas): 9983
por description + title      : 9983


Los dos criterios coinciden, así que son filas repetidas enteras, artefacto del
scraping. Se eliminan antes de cualquier análisis: con el 7,7% de filas
duplicadas, cada promedio, frecuencia y correlación quedaría sesgado.

In [11]:
df = quitar_duplicados(df)

Duplicados eliminados: 9,983 (7.7%) -> 119,988 reseñas


Para las categóricas no sirven media ni desvío. El equivalente son la moda, su
peso sobre el total y la entropía normalizada, que vale 0 si todo cae en una sola
categoría y 1 si se reparte en partes iguales.

In [12]:
resumen_categoricas(df)

,categorías,moda,frec_moda,moda_%,top10_%,entropía_norm
country,43,US,50457,42.1,96.0,0.51
province,425,California,33656,28.1,61.5,0.577
region_1,1229,Napa Valley,4174,4.2,24.6,0.769
region_2,17,Central Coast,10233,21.9,92.3,0.802
variety,707,Pinot Noir,12278,10.2,55.0,0.598
winery,16757,Wines & Winemakers,211,0.2,1.5,0.925
designation,37979,Reserve,1871,2.2,8.6,0.91
taster_name,19,Roger Voss,23560,24.8,92.2,0.797


La entropía separa dos comportamientos que un gráfico de barras deja ver pero
no mide. `country` queda en 0,510, con Estados Unidos solo en el 42,1%, mientras
que `winery` llega a 0,925: 16.757 bodegas en las que las quince más frecuentes
apenas suman el 2%. Esa cola larga significa que ninguna bodega individual es
representativa.

## Bloque 2. Estadística descriptiva

In [13]:
num = df[['points', 'price']]
resumen = num.describe().T
resumen['mediana']   = num.median()
resumen['IQR']       = num.quantile(.75) - num.quantile(.25)
resumen['asimetria'] = num.skew()
resumen['curtosis']  = num.kurtosis()
resumen['CV']        = num.std() / num.mean()
resumen.round(2)

,count,mean,std,min,25%,50%,75%,max,mediana,IQR,asimetria,curtosis,CV
points,119988.0,88.44,3.09,80.0,86.0,88.0,91.0,100.0,88.0,5.0,0.04,-0.34,0.03
price,111593.0,35.62,42.10,4.0,17.0,25.0,42.0,3300.0,25.0,25.0,17.96,809.37,1.18


Dos variables que piden tratamientos opuestos.

`points` tiene asimetría 0,05 y media igual a la mediana, así que media y desvío
sirven. Pero va de 80 a 100 porque sólo se publican reseñas de 80 o más: está
censurada. Decir "el vino promedio saca 88" es sesgo de selección, no un hallazgo.

`price` tiene asimetría 18, curtosis 830 y un máximo que es 132 veces la mediana.
Para esta variable van mediana e IQR, y escala logarítmica en todo gráfico.

In [14]:
for col in ['country', 'variety', 'taster_name', 'province', 'winery']:
    vc = df[col].value_counts()
    print(f'--- {col}: {df[col].nunique()} categorías | '
          f'top-10 concentra {vc.head(10).sum() / vc.sum() * 100:.1f}%')
    print(vc.head(3).to_string(), '\n')

--- country: 43 categorías | top-10 concentra 96.0%
country
US        50457
France    20353
Italy     17940 

--- variety: 707 categorías | top-10 concentra 55.0%
variety
Pinot Noir            12278
Chardonnay            10868
Cabernet Sauvignon     8840 

--- taster_name: 19 categorías | top-10 concentra 92.2%
taster_name
Roger Voss           23560
Michael Schachner    14046
Kerin O’Keefe         9697 

--- province: 425 categorías | top-10 concentra 61.5%
province
California    33656
Washington     7965
Bordeaux       5556 

--- winery: 16757 categorías | top-10 concentra 1.5%
winery
Wines & Winemakers    211
Williams Selyem       204
Testarossa            201 



## Bloque 3. Datos faltantes

Lo que importa no es el porcentaje sino el mecanismo. La mayoría de estos nulos no
son datos perdidos: son campos que no aplican.

In [15]:
# region_2: ¿está incompleta, o completa para el subconjunto donde aplica?
print('no nulos fuera de EE.UU.:', df[df.country.ne('US') & df.region_2.notna()].shape[0])
print('cobertura dentro de EE.UU.:',
      f'{df[df.country.eq("US")].region_2.notna().mean() * 100:.1f}%')

no nulos fuera de EE.UU.: 0
cobertura dentro de EE.UU.: 92.7%


Cero fuera de Estados Unidos y 92,7% adentro. `region_2` designa sub-apelaciones
del sistema estadounidense, así que no es una columna incompleta sino una columna
completa para el subconjunto en el que tiene sentido. Borrarla sería un error.

In [16]:
# ¿El precio falta al azar? (test de MCAR)
por_pais, por_puntaje = diagnostico_faltantes(df)
print(por_pais.head(6).to_string())
print(f'\nsegún puntaje: {por_puntaje.loc[80]}% en 80 -> {por_puntaje.loc[99]}% en 99')

              reseñas  sin_precio_pct
country                              
France          20353            20.0
Austria          3034            16.4
Portugal         5256            14.3
Italy           17940            13.6
South Africa     1301             8.1
New Zealand      1278             3.0

según puntaje: 0.5% en 80 -> 15.2% en 99


Francia tiene cincuenta veces más precios ausentes que Estados Unidos, y la tasa
crece con el puntaje. La ausencia depende de variables observadas, así que el
mecanismo es MAR y no MCAR.

Lo que se sigue de eso: imputar con la mediana global ($25) metería precios de
vino estadounidense barato dentro de Francia e Italia. Hay que condicionar.

In [17]:
# La estrategia se elige midiendo, no por convención:
# se ocultan precios conocidos y se mide el error contra el valor real.
validar_imputacion_precio(df)

,MAE,error_mediano,MAPE_%,sesgo
media global,24.35,17.44,61.11,-2.80
mediana global,21.96,10.00,47.06,-13.25
mediana por país,21.66,10.00,44.44,-13.23
mediana por variedad,19.57,9.00,35.90,-10.97
mediana por país+variedad,18.73,8.00,33.33,-9.98
mediana por país+variedad+puntaje,14.92,6.00,25.00,-6.49


Sumar `points` al agrupador es lo que más aporta: el MAE baja de $21,96 a $14,92
y el MAPE del 47% al 25%. Coincide con la relación exponencial entre precio y
puntaje.

Cada valor imputado queda marcado en `price_imputado`, porque la imputación usa
`points`: analizar la relación precio-puntaje sobre esos valores la reforzaría de
manera artificial. Es circularidad, y la marca permite evitarla.

In [18]:
df = tratar_faltantes(df)
print(f'precios imputados: {df.price_imputado.sum():,}')
print(f'nulos restantes  : {df.isna().sum().sum()}')

precios imputados: 8,395
nulos restantes  : 0


## Bloque 4. Datos atípicos

In [19]:
detectar_atipicos(df)

,price,points
% marcado como atípico,,
IQR clásico,6.16,0.04
z-score |z|>3,0.97,0.10
z modificado (MAD) |z|>3.5,6.41,0.04
IQR sobre log(x),0.97,0.02


Los criterios discrepan por un factor de 6 sobre la misma variable.

El IQR clásico supone simetría, así que sobre `price` marca la cola entera, 6,2%,
y no casos raros. El z-score usa media y desvío, que son justo lo que los atípicos
distorsionan, de modo que se muerde la cola. El z modificado los cambia por
mediana y MAD, que son robustas. Y el IQR sobre el logaritmo aplica el criterio en
la escala donde la variable sí es simétrica.

Ninguno contesta la pregunta que pide la consigna: ¿es un error?

In [20]:
atipicos_contextuales(df).head(8)

,title,province,points,price,mediana_pares,ratio
80290,Château les Ormes Sorbet 2013 Médoc,Bordeaux,88,3300.0,20.0,165.0
120391,Blair 2013 Roger Rose Vineyard Chardonnay (Arr...,California,91,2013.0,40.0,50.3
97150,Domaine Pellé 2014 Morogues Rosé (Menetou-Salon),Loire Valley,87,800.0,20.0,40.0
78235,Burmester 1963 Colheita (Port),Port,87,790.0,25.0,31.6
39627,Henschke 2009 Hill of Grace Shiraz (Eden Valley),South Australia,91,780.0,30.0,26.0
89478,Emmerich Knoll 2013 Ried Loibenberg Smaragd Gr...,Wachau,94,1100.0,50.0,22.0
27518,Vega Sicilia 2008 Unico (Ribera del Duero),Northern Spain,89,500.0,23.0,21.7
76750,Baron Knyphausen 2011 Hattenheimer Wisselbrunn...,Rheingau,90,510.0,28.0,18.2


Comparar cada precio con la mediana de su grupo de pares, o sea los vinos de la
misma provincia y el mismo puntaje, es lo que separa el error del dato válido.

| Vino | Cociente | Veredicto |
|---|---|---|
| Ch. les Ormes Sorbet, $3.300, 88 pts | 165x | Error: único Médoc de 88 pts sobre $50, con mediana de $20 |
| Blair Roger Rose, $2.013, 91 pts | 50x | Error: el precio es el año; el mismo vino 2012 cuesta $28 |
| Château Pétrus, $2.500, 96 pts | 14x | Válido: Pétrus se vende en miles |

Los tres son extremos estadísticamente. Dos son errores y uno es un dato real, así
que la extremidad sola no alcanza para distinguirlos.

In [21]:
impacto_atipicos(df)

,n,Pearson,Spearman
"precio crudo, con atípicos",111593,0.417,0.612
"precio crudo, sin atípicos (IQR)",104724,0.544,0.572
"log(precio), con atípicos",111593,0.618,0.612


El resultado que define el tratamiento.

Pearson sobre precio crudo subestima la relación, 0,417, porque asume linealidad.
Eliminar atípicos la sube a 0,544 y cuesta 6.869 vinos reales. El logaritmo llega
a 0,618 sin tirar un solo dato.

Spearman casi no se mueve entre los tres escenarios, 0,612 / 0,572 / 0,612, porque
opera sobre rangos: sirve como referencia robusta.

Ante la asimetría, transformar la escala gana a eliminar datos.

## Bloque 5. Análisis comparativo

In [22]:
# El año de cosecha no es una columna, pero está en el título.
df = extraer_anio(df)
print(f'año extraído en {df.anio.notna().sum():,} reseñas '
      f'({df.anio.isna().mean() * 100:.1f}% son espumantes "NV" sin cosecha)')
df[['title', 'anio']].sample(4, random_state=1)

año extraído en 115,703 reseñas (3.6% son espumantes "NV" sin cosecha)


,title,anio
108752,Château Bertinerie 2010 Blaye Côtes de Bordeaux,2010
33353,Laudun Chusclan 2013 Camp Romain White (Côtes ...,2013
109555,Bernard Baudry 2010 Les Grézeaux (Chinon),2010
26990,Silver Horse 2011 Sage Red (Paso Robles),2011


La extracción directa falla porque muchas bodegas llevan un año en el nombre, y
un regex sobre el título completo devuelve 1852 para *"Hazlitt 1852 Vineyards 2005
Cabernet"*. Afecta a unos 150 vinos y concentra años falsos en el siglo XIX.

`extraer_anio` recorta primero el nombre de la bodega, que el 100% de los títulos
lleva adelante, y recién entonces busca el año.

In [23]:
analisis_temporal(df).tail(12)

,reseñas,puntaje_medio,precio_mediano
anio,,,
2005,3485,88.35,33.0
2006,5566,88.22,28.0
2007,6733,88.17,28.0
2008,7020,88.21,28.0
2009,9282,88.35,28.0
2010,11290,88.27,28.0
2011,11553,88.27,25.0
2012,14361,88.77,29.0
2013,14327,88.95,28.0


Cuidado con el sesgo de supervivencia. De 1998 sólo se siguen reseñando los
pocos vinos excepcionales que todavía se venden; de 2014 se reseña la producción
corriente. Leer esas medias sin advertirlo daría "los vinos viejos son mejores",
cuando lo que cambió es el filtro por el que entraron a la muestra.

In [24]:
# description es texto libre, pero admite una medida simple: cuánto escribió
# el catador. Es la única variable derivable sin entrar en NLP.
df = agregar_longitud(df)
correlaciones, por_puntaje = analisis_longitud(df)
print(correlaciones.to_string())
print()
print(por_puntaje.to_string())

                     Pearson  Spearman
palabras vs puntaje    0.538     0.528
palabras vs precio     0.350     0.345

        reseñas  palabras_medias
points                          
80          395             26.2
81          677             26.8
82         1750             28.1
83         2830             31.4
84         5860             32.7
85         8470             34.4
86        10748             36.0
87        14225             37.7
88        14209             39.4
89        10307             41.4
90        12920             43.2
91         9806             45.5
92         8244             47.7
93         5634             49.9
94         3328             52.7
95         1387             55.8
96          481             58.5
97          206             59.5
98           69             62.3
99           28             64.5
100          19             69.5


La extensión de la nota crece de forma monótona con el puntaje, sin una sola
excepción: 26 palabras en los vinos de 80 puntos y 70 en los de 100. Un vino de
puntaje máximo recibe una reseña dos veces y media más larga que uno del piso de
publicación.

La correlación de 0,538 es más alta que la del precio crudo con el puntaje. Admite
dos lecturas que estos datos no permiten separar: que un vino más complejo da más
material para describir, o que el catador se explaya cuando algo le gusta. Con el
precio la correlación es bastante menor, 0,350, lo que sugiere que la extensión
sigue a la valoración más que al costo.

In [25]:
efecto_catador(df)

,reseñas,media_cruda,comparables,desvio_controlado
taster_name,,,,
Anne Krebiehl MW,3290,90.63,2155.0,0.76
Matt Kettmann,5730,90.06,5082.0,0.85
Virginie Boone,8708,89.22,8100.0,0.20
Paul Gregutt,8868,89.09,8001.0,-0.00
Kerin O’Keefe,9697,88.90,6519.0,0.01
Sean P. Sullivan,4461,88.75,3998.0,-0.17
Roger Voss,23560,88.73,20728.0,-0.04
Jim Gordon,3766,88.60,3276.0,-0.24
Joe Czerwinski,4766,88.52,3061.0,-0.22


El hallazgo más fuerte del trabajo.

Las medias crudas muestran una brecha de 3,77 puntos entre el catador más severo
(86,86) y la más generosa (90,63). Pero los catadores no reciben vinos al azar:
Schachner reseña 43% España y 29% Chile, mientras que Krebiehl cubre 60% Austria y
38% Francia.

Midiendo cuánto se aparta cada reseña del promedio de su propio grupo de país y
variedad, la brecha cae a 0,78 y el supuesto severo queda en -0,02, o sea
exactamente en el promedio de los vinos que evalúa.

Casi toda la diferencia venía de qué vinos le tocaban. Es confusión por una
variable omitida, y comparar las medias crudas habría llevado a una conclusión
equivocada sobre el trabajo de personas con nombre y apellido.

In [26]:
mejor_relacion_calidad_precio(df)

,reseñas,puntaje_mediano,precio_mediano,puntos_por_dolar
variety,,,,
Portuguese White,905,87.0,13.0,6.69
Torrontés,231,86.0,13.0,6.62
Verdejo,264,86.5,14.0,6.18
Moscato,302,86.0,14.0,6.14
Garnacha,307,86.0,14.0,6.14
Melon,220,88.0,15.0,5.87
Carmenère,525,87.0,15.0,5.80
Pinot Grigio,934,86.0,15.0,5.73
Rosé,2951,87.0,16.0,5.44


---

## Resumen

1. El archivo llegó con tres corrupciones que ninguna carga convencional señala.
2. El enunciado se equivoca en columnas (10 contra 13) y reseñas (130.000 contra 129.971).
3. El 7,7% de las filas estaba duplicado.
4. `price` es MAR y no MCAR, cosa que se probó en vez de asumirse.
5. La imputación se eligió por validación: el error baja de $21,96 a $14,92.
6. Extremo estadístico no es lo mismo que error: hacen falta los pares para distinguirlos.
7. Ante asimetría, transformar gana a eliminar: 0,417 sube a 0,618 sin perder datos.
8. La extensión de la nota acompaña al puntaje con una correlación de 0,538.
9. La severidad aparente de un catador (-1,6) era asignación de vinos (-0,02 controlado).